# Large-N Accuracy Benchmarks


## Large-N Runtime Toggle
These benchmarks now support the dedicated `large_n` runtime path. Set `GPU_DEVICE = "9"` and `RUNTIME_PATH = "large_n"` below for the new path, or switch `RUNTIME_PATH` to `"legacy"` for comparison.

In [ ]:
import os
GPU_DEVICE = "9"
RUNTIME_PATH = "large_n"  # or "legacy" for comparison
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_DEVICE
print(f"CUDA_VISIBLE_DEVICES={GPU_DEVICE}, RUNTIME_PATH={RUNTIME_PATH}")

def inject_runtime_path(kwargs):
    updated = dict(kwargs)
    updated["runtime_path"] = RUNTIME_PATH
    return updated


## Goals

- Measure solver accuracy against direct-sum references across opening angles, orders, and MAC choices.
- Validate operator-level accuracy independently of the full tree traversal.
- Keep the notebook focused on numerical quality rather than runtime profiling.


In [ ]:
import os

# --- Option A: automatic GPU selection with autocvd ---
USE_AUTOCVD = False
AUTOCVD_NUM_GPUS = 1
AUTOCVD_LEAST_USED = True
AUTOCVD_EXCLUDE = []

# --- Option B: manual selection (set to string like '0' or '0,1') ---
MANUAL_CUDA_VISIBLE_DEVICES = GPU_DEVICE  # None #"1,2,3,4,5,6,7,8,9"#None

if MANUAL_CUDA_VISIBLE_DEVICES is not None:
    os.environ["CUDA_VISIBLE_DEVICES"] = MANUAL_CUDA_VISIBLE_DEVICES
    print("Set CUDA_VISIBLE_DEVICES =", os.environ["CUDA_VISIBLE_DEVICES"])
elif USE_AUTOCVD:
    try:
        from autocvd import autocvd

        autocvd(
            num_gpus=AUTOCVD_NUM_GPUS,
            least_used=AUTOCVD_LEAST_USED,
            exclude=AUTOCVD_EXCLUDE,
        )
        print(
            "autocvd selected CUDA_VISIBLE_DEVICES =",
            os.environ.get("CUDA_VISIBLE_DEVICES"),
        )
    except ImportError:
        print(
            "autocvd is not installed. Install it or set MANUAL_CUDA_VISIBLE_DEVICES."
        )
else:
    print(
        "Using existing CUDA visibility:",
        os.environ.get("CUDA_VISIBLE_DEVICES", "<all visible>"),
    )

# Index precision switch (must be set before importing JAX/jaccpot/yggdrax).
INDEX_PRECISION = "int32"  # choose from: "int32", "int64"
os.environ.setdefault("JACCPOT_INDEX_PRECISION", INDEX_PRECISION)
os.environ.setdefault("YGGDRAX_INDEX_PRECISION", INDEX_PRECISION)
print("Index precision:", os.environ.get("JACCPOT_INDEX_PRECISION"))

# GPU memory-fragmentation / graph-memory safeguards.
os.environ.setdefault("TF_GPU_ALLOCATOR", "cuda_malloc_async")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_ALLOCATOR", "platform")
os.environ.setdefault("JACCPOT_PREPARE_DIAGNOSTICS", "1")
os.environ.setdefault("YGGDRAX_TRAVERSAL_DIAGNOSTICS", "1")
if "--xla_gpu_enable_command_buffer=" not in os.environ.get("XLA_FLAGS", ""):
    existing_xla_flags = os.environ.get("XLA_FLAGS", "").strip()
    command_buffer_off = "--xla_gpu_enable_command_buffer="
    os.environ["XLA_FLAGS"] = (
        f"{existing_xla_flags} {command_buffer_off}".strip()
        if existing_xla_flags
        else command_buffer_off
    )

# Runtime cache limits (reduce transient cache memory pressure for capacity runs).
os.environ.setdefault("JACCPOT_OPERATOR_CACHE_MAX", "256")
os.environ.setdefault("JACCPOT_GROUPED_OPERATOR_CACHE_MAX", "16")
os.environ.setdefault("JACCPOT_GROUPED_SEGMENT_CACHE_MAX", "16")
os.environ.setdefault(
    "JACCPOT_GROUPED_OPERATOR_CACHE_ENTRY_MAX_BYTES", str(16 * 1024 * 1024)
)
os.environ.setdefault(
    "JACCPOT_GROUPED_OPERATOR_CACHE_TOTAL_MAX_BYTES", str(64 * 1024 * 1024)
)
os.environ.setdefault(
    "JACCPOT_GROUPED_SEGMENT_CACHE_ENTRY_MAX_BYTES", str(8 * 1024 * 1024)
)
os.environ.setdefault(
    "JACCPOT_GROUPED_SEGMENT_CACHE_TOTAL_MAX_BYTES", str(32 * 1024 * 1024)
)
print(
    "Runtime cache limits:",
    {
        "op_max": os.environ.get("JACCPOT_OPERATOR_CACHE_MAX"),
        "grp_op_max": os.environ.get("JACCPOT_GROUPED_OPERATOR_CACHE_MAX"),
        "grp_seg_max": os.environ.get("JACCPOT_GROUPED_SEGMENT_CACHE_MAX"),
        "grp_op_total_mb": int(
            os.environ.get("JACCPOT_GROUPED_OPERATOR_CACHE_TOTAL_MAX_BYTES", "0")
        )
        // (1024 * 1024),
        "grp_seg_total_mb": int(
            os.environ.get("JACCPOT_GROUPED_SEGMENT_CACHE_TOTAL_MAX_BYTES", "0")
        )
        // (1024 * 1024),
    },
)

VISIBLE_PHYSICAL_GPUS = [
    part.strip()
    for part in os.environ.get("CUDA_VISIBLE_DEVICES", "").split(",")
    if part.strip() != ""
]
NVIDIA_SMI_GPU_INDEX = int(VISIBLE_PHYSICAL_GPUS[0]) if VISIBLE_PHYSICAL_GPUS else 0
print("JACCPOT_PREPARE_DIAGNOSTICS:", os.environ.get("JACCPOT_PREPARE_DIAGNOSTICS"))
print("YGGDRAX_TRAVERSAL_DIAGNOSTICS:", os.environ.get("YGGDRAX_TRAVERSAL_DIAGNOSTICS"))
print("nvidia-smi physical GPU index:", NVIDIA_SMI_GPU_INDEX)

In [ ]:
import pathlib
import sys
from dataclasses import replace

import jax

# Force float64 in accuracy sweeps; otherwise JAX may silently downcast to float32.
jax.config.update("jax_enable_x64", True)

import jax.numpy as jnp
import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = pathlib.Path.cwd().resolve()
if not (REPO_ROOT / "jaccpot").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

from jaccpot import (
    FMMAdvancedConfig,
    FMMPreset,
    FarFieldConfig,
    FastMultipoleMethod,
    NearFieldConfig,
    RuntimePolicyConfig,
    TreeConfig,
 )
from yggdrax import Tree, compute_tree_geometry
from yggdrax.interactions import DualTreeTraversalConfig, build_interactions_and_neighbors
from examples import benchmark_utils as bench_utils

#plt.style.use("seaborn-v0_8-darkgrid")
import inspect

In [ ]:
import inspect
import jaccpot
import jaccpot.runtime._fmm_impl as jf
import yggdrax
import yggdrax._interactions_impl as yi

print("jaccpot:", jaccpot.__file__)
print("yggdrax:", yggdrax.__file__)

src_j = inspect.getsource(jf.FastMultipoleMethod._resolve_runtime_execution_overrides)
print("streamed/grouped guard:", "if self.streamed_far_pairs and grouped_interactions" in src_j)
print("minimum-memory traversal None:", "traversal_config = None" in src_j)

src_y = inspect.getsource(yi)
print("count-pass fill cap:", "_COUNT_PASS_FILL_INTERACTION_CAP = 4096" in src_y)


## Benchmark configuration

Adjust the parameters below to control sweep ranges, repetition counts, and floating-point precision.

In [ ]:
runtime_particle_counts = [
    512,
    1024,
    2048,
    4096,
    8192,
    16384,
    32768,
    65536,
    131072,
    262144,
    524288,
    1048576,
    2097152,
    4194304,
    8388608,
    16777216
 ]
runtime_leaf_size = 128
runtime_max_order = 4
runtime_runs = 3
runtime_warmup = 1
runtime_isolate_process_per_n = True
runtime_worker_benchmark_scope = "full"
runtime_worker_autotune_cache_path = REPO_ROOT / "benchmarks" / "runtime_accuracy_worker_autotune_cache.json"

accuracy_particle_count = 4096 #16384  # keep direct reference tractable
accuracy_leaf_size = 128
accuracy_orders = [1, 2, 3, 4]
accuracy_thetas = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 0.9]

# New: order-only accuracy sweep at fixed θ (clean scaling curve)
order_sweep_particle_count = 4096 #16384
order_sweep_leaf_size = 128
order_sweep_orders = list(range(1, 9))
order_sweep_theta = 0.4
order_sweep_mac_type = "dehnen"

softening = 1e-3
runtime_working_dtype = jnp.float32
accuracy_working_dtype = jnp.float64  # tighten accuracy sweep floor
runtime_key = jax.random.PRNGKey(0)
accuracy_key = jax.random.PRNGKey(1)
order_diag_key = jax.random.PRNGKey(2)

# Lean large-N runtime baseline aligned with the production H100/H200 sweep.
runtime_mac_type = "engblom"
runtime_memory_first_m2l_chunk_size = 1024


def runtime_traversal_seed_for_num_particles(num_particles):
    del num_particles
    return dict(
        max_pair_queue=262144,
        process_block=256,
        max_interactions_per_node=8192,
        max_neighbors_per_leaf=4096,
    )


runtime_baseline_num_particles = max(int(v) for v in runtime_particle_counts)
runtime_baseline_traversal = runtime_traversal_seed_for_num_particles(runtime_baseline_num_particles)
BEST_TRAVERSAL_CONFIG = DualTreeTraversalConfig(**runtime_baseline_traversal)

runtime_advanced = FMMAdvancedConfig(
    tree=TreeConfig(
        tree_type="radix",
        mode="lbvh",
        leaf_target=runtime_leaf_size,
        refine_local=False,
        max_refine_levels=0,
        aspect_threshold=16.0,
    ),
    farfield=FarFieldConfig(
        rotation="solidfmm",
        mode="pair_grouped",
        grouped_interactions=False,
        streamed_far_pairs=True,
        m2l_chunk_size=int(runtime_memory_first_m2l_chunk_size),
    ),
    nearfield=NearFieldConfig(
        mode="bucketed",
        edge_chunk_size=128,
        precompute_scatter_schedules=False,
    ),
    runtime=RuntimePolicyConfig(
        host_refine_mode="off",
        fail_fast=True,
        jit_tree=True,
        jit_traversal=True,
        memory_objective="minimum_memory",
        max_pair_queue=int(runtime_baseline_traversal["max_pair_queue"]),
        pair_process_block=None,
        traversal_config=BEST_TRAVERSAL_CONFIG,
        enable_interaction_cache=False,
        retain_traversal_result=False,
        retain_interactions=False,
        autotune_m2l_chunk=False,
        upward_leaf_batch_size=2048,
    ),
    mac_type=runtime_mac_type,
)
runtime_fmm_kwargs = dict(
    preset=FMMPreset.LARGE_N_GPU,
    basis="solidfmm",
    theta=0.6,
    softening=softening,
    working_dtype=runtime_working_dtype,
    advanced=runtime_advanced,
)

accuracy_advanced = FMMAdvancedConfig(
    tree=TreeConfig(leaf_target=accuracy_leaf_size),
    farfield=FarFieldConfig(
        rotation="solidfmm",
        mode="pair_grouped",
        grouped_interactions=True,
        m2l_chunk_size=int(runtime_memory_first_m2l_chunk_size),
    ),
    nearfield=NearFieldConfig(mode="bucketed", edge_chunk_size=128),
    runtime=RuntimePolicyConfig(
        max_pair_queue=262144,
        pair_process_block=256,
        traversal_config=BEST_TRAVERSAL_CONFIG,
    ),
    mac_type="dehnen",
)
accuracy_fmm_kwargs = dict(
    preset=FMMPreset.BALANCED,
    basis="solidfmm",
    theta=0.6,
    softening=softening,
    working_dtype=accuracy_working_dtype,
    advanced=accuracy_advanced,
)


def override_fmm_kwargs(
    fmm_kwargs,
    *,
    theta=None,
    farfield_mode=None,
    grouped_interactions=None,
    nearfield_mode=None,
    nearfield_edge_chunk_size=None,
    mac_type=None,
):
    kwargs = dict(fmm_kwargs)
    advanced = kwargs["advanced"]

    farfield = advanced.farfield
    if farfield_mode is not None or grouped_interactions is not None:
        farfield = replace(
            farfield,
            mode=farfield.mode if farfield_mode is None else farfield_mode,
            grouped_interactions=(
                farfield.grouped_interactions
                if grouped_interactions is None
                else bool(grouped_interactions)
            ),
        )

    nearfield = advanced.nearfield
    if nearfield_mode is not None or nearfield_edge_chunk_size is not None:
        nearfield = replace(
            nearfield,
            mode=nearfield.mode if nearfield_mode is None else nearfield_mode,
            edge_chunk_size=(
                nearfield.edge_chunk_size
                if nearfield_edge_chunk_size is None
                else int(nearfield_edge_chunk_size)
            ),
        )

    next_mac_type = advanced.mac_type if mac_type is None else str(mac_type)
    kwargs["advanced"] = replace(
        advanced,
        farfield=farfield,
        nearfield=nearfield,
        mac_type=next_mac_type,
    )
    if theta is not None:
        kwargs["theta"] = float(theta)
    return kwargs

runtime_fmm_probe = FastMultipoleMethod(**runtime_fmm_kwargs)
print("runtime memory_objective:", runtime_fmm_probe._impl.memory_objective)
print("runtime fail_fast:", runtime_fmm_probe._impl.fail_fast)
print("runtime mac_type:", runtime_fmm_probe.advanced.mac_type)
print(
    f"runtime overrides @ N={runtime_baseline_num_particles}:",
    runtime_fmm_probe._impl._resolve_runtime_execution_overrides(num_particles=int(runtime_baseline_num_particles)),
)

In [ ]:
def direct_accelerations(positions, masses, *, softening, G=1.0):
    """Reference O(N^2) direct-sum accelerations."""
    positions = jnp.asarray(positions)
    masses = jnp.asarray(masses)
    diff = positions[:, None, :] - positions[None, :, :]
    dist_sq = jnp.sum(diff**2, axis=-1) + softening**2
    mask = ~jnp.eye(positions.shape[0], dtype=bool)
    inv_r3 = jnp.where(mask, 1.0 / (dist_sq * jnp.sqrt(dist_sq)), 0.0)
    weighted = diff * masses[None, :, None] * inv_r3[..., None]
    return -G * jnp.sum(weighted, axis=1)


def relative_l2_error(estimate, reference):
    estimate = jnp.asarray(estimate)
    reference = jnp.asarray(reference)
    numerator = jnp.linalg.norm(estimate - reference)
    denominator = jnp.linalg.norm(reference)
    return float(numerator / denominator)


def relative_max_error(estimate, reference):
    estimate = jnp.asarray(estimate)
    reference = jnp.asarray(reference)
    numerator = jnp.max(jnp.linalg.norm(estimate - reference, axis=-1))
    denominator = jnp.max(jnp.linalg.norm(reference, axis=-1))
    return float(numerator / denominator)


def _evaluate_prepared_kwargs(fmm):
    params = inspect.signature(fmm.evaluate_prepared_state).parameters
    if "jit_traversal" in params:
        return {"jit_traversal": True}
    return {}
def sweep_runtimes(
    particle_counts,
    *,
    leaf_size,
    max_order,
    runs,
    warmup,
    dtype,
    key,
    fmm_kwargs,
    strict=False,
 ):
    records = []
    fmm = FastMultipoleMethod(**fmm_kwargs)
    current_key = key
    for num_particles in particle_counts:
        try:
            positions, masses, current_key = bench_utils.generate_random_distribution(
                num_particles,
                key=current_key,
                dtype=dtype,
            )

            full_timing = bench_utils.time_callable(
                fmm.compute_accelerations,
                positions,
                masses,
                leaf_size=leaf_size,
                max_order=max_order,
                reuse_prepared_state=False,
                warmup=warmup,
                runs=runs,
            )

            prepare_once_timing = bench_utils.time_callable(
                fmm.prepare_state,
                positions,
                masses,
                leaf_size=leaf_size,
                max_order=max_order,
                warmup=warmup,
                runs=runs,
            )
            prepared_state = prepare_once_timing.result

            eval_kwargs = _evaluate_prepared_kwargs(fmm)
            eval_timing = bench_utils.time_callable(
                fmm.evaluate_prepared_state,
                prepared_state,
                warmup=warmup,
                runs=runs,
                **eval_kwargs,
            )

            records.append(
                {
                    "num_particles": num_particles,
                    "mean_seconds": full_timing.mean,
                    "std_seconds": full_timing.std,
                    "prepare_mean_seconds": prepare_once_timing.mean,
                    "prepare_std_seconds": prepare_once_timing.std,
                    "evaluate_mean_seconds": eval_timing.mean,
                    "evaluate_std_seconds": eval_timing.std,
                    "error": "",
                }
            )
        except Exception as exc:
            msg = f"{type(exc).__name__}: {exc}"
            print(f"[sweep_runtimes] N={num_particles} failed: {msg}")
            records.append(
                {
                    "num_particles": num_particles,
                    "mean_seconds": float("nan"),
                    "std_seconds": float("nan"),
                    "prepare_mean_seconds": float("nan"),
                    "prepare_std_seconds": float("nan"),
                    "evaluate_mean_seconds": float("nan"),
                    "evaluate_std_seconds": float("nan"),
                    "error": msg,
                }
            )
            if strict:
                raise
    return pd.DataFrame(records)


def sweep_prepared_eval_runtimes(
    particle_counts,
    *,
    leaf_size,
    max_order,
    runs,
    warmup,
    dtype,
    key,
    fmm_kwargs,
    strict=False,
 ):
    records = []
    fmm = FastMultipoleMethod(**fmm_kwargs)
    current_key = key
    for num_particles in particle_counts:
        try:
            positions, masses, current_key = bench_utils.generate_random_distribution(
                num_particles,
                key=current_key,
                dtype=dtype,
            )
            state = fmm.prepare_state(
                positions,
                masses,
                leaf_size=leaf_size,
                max_order=max_order,
            )
            eval_kwargs = _evaluate_prepared_kwargs(fmm)

            eval_timing = bench_utils.time_callable(
                fmm.evaluate_prepared_state,
                state,
                warmup=warmup,
                runs=runs,
                **eval_kwargs,
            )

            eval_jit = jax.jit(
                lambda st: fmm.evaluate_prepared_state(st, **eval_kwargs)
            )
            # Trigger compilation outside measured runs.
            bench_utils.time_callable(
                eval_jit,
                state,
                warmup=0,
                runs=1,
            )
            eval_jit_timing = bench_utils.time_callable(
                eval_jit,
                state,
                warmup=warmup,
                runs=runs,
            )

            records.append(
                {
                    "num_particles": num_particles,
                    "prepared_eval_mean_seconds": eval_timing.mean,
                    "prepared_eval_std_seconds": eval_timing.std,
                    "prepared_eval_jit_mean_seconds": eval_jit_timing.mean,
                    "prepared_eval_jit_std_seconds": eval_jit_timing.std,
                    "error": "",
                }
            )
        except Exception as exc:
            msg = f"{type(exc).__name__}: {exc}"
            print(f"[sweep_prepared_eval_runtimes] N={num_particles} failed: {msg}")
            records.append(
                {
                    "num_particles": num_particles,
                    "prepared_eval_mean_seconds": float("nan"),
                    "prepared_eval_std_seconds": float("nan"),
                    "prepared_eval_jit_mean_seconds": float("nan"),
                    "prepared_eval_jit_std_seconds": float("nan"),
                    "error": msg,
                }
            )
            if strict:
                raise
    return pd.DataFrame(records)


def profile_prepare_components(
    particle_counts,
    *,
    leaf_size,
    max_order,
    dtype,
    key,
    fmm_kwargs,
    runs,
    warmup,
    strict=False,
):
    records = []
    fmm = FastMultipoleMethod(**fmm_kwargs)
    current_key = key

    tree_type = str(getattr(fmm._impl, "tree_type", "radix"))
    tree_mode = str(getattr(fmm._impl, "tree_build_mode", "lbvh"))
    ygg_build_mode = "fixed_depth" if tree_mode == "fixed_depth" else "adaptive"
    theta_val = float(getattr(fmm._impl, "theta", fmm_kwargs.get("theta", 0.6)))
    traversal_cfg = fmm.advanced.runtime.traversal_config
    mac_type = str(getattr(fmm, "mac_type", "dehnen"))
    dehnen_radius_scale = float(getattr(fmm._impl, "dehnen_radius_scale", 1.0))

    for num_particles in particle_counts:
        try:
            positions, masses, current_key = bench_utils.generate_random_distribution(
                num_particles,
                key=current_key,
                dtype=dtype,
            )

            tree_timing = bench_utils.time_callable(
                Tree.from_particles,
                positions,
                masses,
                tree_type=tree_type,
                build_mode=ygg_build_mode,
                return_reordered=True,
                leaf_size=int(leaf_size),
                warmup=warmup,
                runs=runs,
            )

            tree = Tree.from_particles(
                positions,
                masses,
                tree_type=tree_type,
                build_mode=ygg_build_mode,
                return_reordered=True,
                leaf_size=int(leaf_size),
            )
            geometry = compute_tree_geometry(tree, tree.positions_sorted)

            interactions_timing = bench_utils.time_callable(
                build_interactions_and_neighbors,
                tree,
                geometry,
                theta=theta_val,
                traversal_config=traversal_cfg,
                mac_type=mac_type,
                dehnen_radius_scale=dehnen_radius_scale,
                warmup=warmup,
                runs=runs,
            )

            prepare_timing = bench_utils.time_callable(
                fmm.prepare_state,
                positions,
                masses,
                leaf_size=int(leaf_size),
                max_order=max_order,
                warmup=warmup,
                runs=runs,
            )

            residual = max(
                float(prepare_timing.mean)
                - float(tree_timing.mean)
                - float(interactions_timing.mean),
                0.0,
            )
            records.append(
                {
                    "num_particles": num_particles,
                    "tree_build_mean_seconds": float(tree_timing.mean),
                    "interactions_mean_seconds": float(interactions_timing.mean),
                    "upward_mean_seconds": residual,
                    "downward_mean_seconds": 0.0,
                    "prepare_component_sum_seconds": float(prepare_timing.mean),
                    "error": "",
                }
            )
        except Exception as exc:
            msg = f"{type(exc).__name__}: {exc}"
            print(f"[profile_prepare_components] N={num_particles} failed: {msg}")
            records.append(
                {
                    "num_particles": num_particles,
                    "tree_build_mean_seconds": float("nan"),
                    "upward_mean_seconds": float("nan"),
                    "interactions_mean_seconds": float("nan"),
                    "downward_mean_seconds": float("nan"),
                    "prepare_component_sum_seconds": float("nan"),
                    "error": msg,
                }
            )
            if strict:
                raise

    return pd.DataFrame(records)


def sweep_accuracy(
    positions,
    masses,
    *,
    thetas,
    orders,
    leaf_size,
    fmm_kwargs,
 ):
    reference = direct_accelerations(
        positions,
        masses,
        softening=fmm_kwargs.get("softening", 0.0),
    )
    rows = []

    # Reuse one solver per theta and evaluate through prepared-state API.
    for theta in thetas:
        theta_kwargs = dict(fmm_kwargs)
        theta_kwargs["theta"] = theta
        fmm = FastMultipoleMethod(**theta_kwargs)
        if fmm.basis != "solidfmm" or fmm.complex_rotation != "solidfmm":
            raise ValueError(
                "sweep_accuracy must use basis='solidfmm' and solidfmm rotation convention"
            )

        for order in orders:
            state = fmm.prepare_state(
                positions,
                masses,
                leaf_size=leaf_size,
                max_order=order,
            )
            accelerations = fmm.evaluate_prepared_state(state)
            rows.append(
                {
                    "theta": float(theta),
                    "max_order": int(order),
                    "relative_l2_error": relative_l2_error(accelerations, reference),
                    "relative_max_error": relative_max_error(accelerations, reference),
                }
            )

    return pd.DataFrame(rows)


In [ ]:
import gc
import json
import subprocess


def _release_runtime_memory(fmm):
    clear_fn = getattr(fmm, "clear_runtime_caches", None)
    if callable(clear_fn):
        clear_fn(clear_jax_compilation=True)
        return
    if hasattr(fmm, "clear_prepared_state_cache"):
        fmm.clear_prepared_state_cache()
    jax.clear_caches()


def _iter_notebook_fmm_instances(obj, *, seen=None):
    if seen is None:
        seen = set()
    obj_id = id(obj)
    if obj_id in seen:
        return
    seen.add(obj_id)
    if isinstance(obj, FastMultipoleMethod):
        yield obj
        return
    if isinstance(obj, dict):
        for value in obj.values():
            yield from _iter_notebook_fmm_instances(value, seen=seen)
        return
    if isinstance(obj, (list, tuple, set)):
        for value in obj:
            yield from _iter_notebook_fmm_instances(value, seen=seen)


def _release_notebook_runtime_state():
    released = set()
    for name, value in list(globals().items()):
        if name.startswith("__"):
            continue
        for fmm in _iter_notebook_fmm_instances(value):
            fmm_id = id(fmm)
            if fmm_id in released:
                continue
            try:
                _release_runtime_memory(fmm)
            except Exception:
                pass
            released.add(fmm_id)
    gc.collect()
    jax.clear_caches()


def _serialize_fmm_kwargs_for_worker(fmm_kwargs):
    probe_fmm = FastMultipoleMethod(**fmm_kwargs)
    advanced = probe_fmm.advanced
    traversal_cfg = advanced.runtime.traversal_config
    traversal_payload = None
    if traversal_cfg is not None:
        traversal_payload = {
            "process_block": int(traversal_cfg.process_block),
            "max_neighbors_per_leaf": int(traversal_cfg.max_neighbors_per_leaf),
            "max_interactions_per_node": int(traversal_cfg.max_interactions_per_node),
            "max_pair_queue": int(traversal_cfg.max_pair_queue),
        }
    preset_value = fmm_kwargs.get("preset", "fast")
    if hasattr(preset_value, "value"):
        preset_value = preset_value.value
    payload = {
        "preset": str(preset_value),
        "basis": str(fmm_kwargs.get("basis", "solidfmm")),
        "theta": float(fmm_kwargs.get("theta", 0.6)),
        "softening": float(fmm_kwargs.get("softening", 1e-3)),
        "working_dtype": str(jnp.dtype(getattr(probe_fmm._impl, "working_dtype", jnp.float32))),
        "tree_type": str(advanced.tree.tree_type),
        "leaf_target": int(advanced.tree.leaf_target),
        "farfield_rotation": str(advanced.farfield.rotation),
        "farfield_mode": str(advanced.farfield.mode),
        "grouped_interactions": bool(advanced.farfield.grouped_interactions),
        "streamed_far_pairs": advanced.farfield.streamed_far_pairs,
        "mixed_order": bool(advanced.farfield.mixed_order),
        "mixed_order_min_order": advanced.farfield.mixed_order_min_order,
        "nearfield_mode": str(advanced.nearfield.mode),
        "nearfield_edge_chunk_size": int(advanced.nearfield.edge_chunk_size),
        "precompute_scatter_schedules": bool(advanced.nearfield.precompute_scatter_schedules),
        "pair_process_block": (
            None
            if advanced.runtime.pair_process_block is None
            else int(advanced.runtime.pair_process_block)
        ),
        "fail_fast": bool(advanced.runtime.fail_fast),
        "memory_objective": str(advanced.runtime.memory_objective),
        "jit_traversal": bool(advanced.runtime.jit_traversal),
        "traversal_config": traversal_payload,
        "enable_interaction_cache": bool(advanced.runtime.enable_interaction_cache),
        "retain_traversal_result": bool(advanced.runtime.retain_traversal_result),
        "retain_interactions": bool(advanced.runtime.retain_interactions),
        "autotune_m2l_chunk": bool(advanced.runtime.autotune_m2l_chunk),
        "adaptive_order": bool(getattr(probe_fmm._impl, "adaptive_order", False)),
        "p_gears": [int(v) for v in getattr(probe_fmm._impl, "p_gears", tuple())],
        "adaptive_error_model": str(getattr(probe_fmm._impl, "adaptive_error_model", "tail_proxy")),
        "adaptive_eps": (
            None
            if getattr(probe_fmm._impl, "adaptive_eps", None) is None
            else float(getattr(probe_fmm._impl, "adaptive_eps"))
        ),
        "mac_force_scale_mode": str(getattr(probe_fmm._impl, "mac_force_scale_mode", "prev")),
        "mac_type": str(advanced.mac_type),
        "benchmark_scope": str(globals().get("runtime_worker_benchmark_scope", "full")),
    }
    _release_runtime_memory(probe_fmm)
    return payload


def _build_worker_command(mode, *, num_particles, leaf_size, max_order, runs, warmup, dtype, fmm_kwargs):
    worker_script = REPO_ROOT / "examples" / "benchmark_gpu_radix_worker.py"
    payload = _serialize_fmm_kwargs_for_worker(fmm_kwargs)
    seed = int(jax.device_get(jnp.asarray(runtime_key))[0]) if "runtime_key" in globals() else 0
    cmd = [
        sys.executable,
        str(worker_script),
        "--mode",
        str(mode),
        "--num-particles",
        str(int(num_particles)),
        "--leaf-size",
        str(int(leaf_size)),
        "--max-order",
        str(int(max_order)),
        "--runs",
        str(int(runs)),
        "--warmup",
        str(int(warmup)),
        "--dtype",
        str(jnp.dtype(dtype)),
        "--seed",
        str(seed),
        "--autotune-cache",
        str(globals().get("runtime_worker_autotune_cache_path", REPO_ROOT / "benchmarks" / "runtime_accuracy_worker_autotune_cache.json")),
        "--config-json",
        json.dumps(payload),
    ]
    return cmd


def _parse_worker_json_output(lines):
    filtered = [line.strip() for line in lines if line and line.strip()]
    if not filtered:
        raise RuntimeError("worker produced no output")
    for line in reversed(filtered):
        if line == "__JACCPOT_WORKER_READY__":
            continue
        try:
            return json.loads(line)
        except json.JSONDecodeError:
            continue
    raise RuntimeError("worker produced no JSON output")


def _run_worker_case(mode, *, num_particles, leaf_size, max_order, runs, warmup, dtype, fmm_kwargs):
    _release_notebook_runtime_state()
    cmd = _build_worker_command(
        mode,
        num_particles=num_particles,
        leaf_size=leaf_size,
        max_order=max_order,
        runs=runs,
        warmup=warmup,
        dtype=dtype,
        fmm_kwargs=fmm_kwargs,
    )
    result = subprocess.run(cmd, check=False, capture_output=True, text=True)
    _release_notebook_runtime_state()
    if result.returncode != 0:
        details = (result.stderr or result.stdout or "").strip()
        raise RuntimeError(f"worker failed (exit={result.returncode}): {details}")
    return _parse_worker_json_output(result.stdout.splitlines())


def sweep_runtimes(
    particle_counts,
    *,
    leaf_size,
    max_order,
    runs,
    warmup,
    dtype,
    key,
    fmm_kwargs,
    strict=False,
):
    del key
    records = []
    use_subprocess = bool(globals().get("runtime_isolate_process_per_n", False))
    fmm = None if use_subprocess else FastMultipoleMethod(**fmm_kwargs)
    current_key = runtime_key if "runtime_key" in globals() else None
    for num_particles in particle_counts:
        try:
            if use_subprocess:
                out = _run_worker_case(
                    "sweep",
                    num_particles=int(num_particles),
                    leaf_size=int(leaf_size),
                    max_order=int(max_order),
                    runs=int(runs),
                    warmup=int(warmup),
                    dtype=dtype,
                    fmm_kwargs=fmm_kwargs,
                )
                records.append(
                    {
                        "num_particles": int(num_particles),
                        "mean_seconds": float(out.get("mean_seconds", float("nan"))),
                        "std_seconds": float(out.get("std_seconds", float("nan"))),
                        "prepare_mean_seconds": float(out.get("prepare_mean_seconds", float("nan"))),
                        "prepare_std_seconds": float(out.get("prepare_std_seconds", float("nan"))),
                        "evaluate_mean_seconds": float(out.get("evaluate_mean_seconds", float("nan"))),
                        "evaluate_std_seconds": float(out.get("evaluate_std_seconds", float("nan"))),
                        "error": str(out.get("error", "")),
                    }
                )
                continue

            positions, masses, current_key = bench_utils.generate_random_distribution(
                num_particles,
                key=current_key,
                dtype=dtype,
            )
            full_timing = bench_utils.time_callable(
                fmm.compute_accelerations,
                positions,
                masses,
                leaf_size=leaf_size,
                max_order=max_order,
                reuse_prepared_state=False,
                warmup=warmup,
                runs=runs,
            )
            prepare_once_timing = bench_utils.time_callable(
                fmm.prepare_state,
                positions,
                masses,
                leaf_size=leaf_size,
                max_order=max_order,
                warmup=warmup,
                runs=runs,
            )
            prepared_state = prepare_once_timing.result
            eval_kwargs = _evaluate_prepared_kwargs(fmm)
            eval_timing = bench_utils.time_callable(
                fmm.evaluate_prepared_state,
                prepared_state,
                warmup=warmup,
                runs=runs,
                **eval_kwargs,
            )
            records.append(
                {
                    "num_particles": num_particles,
                    "mean_seconds": full_timing.mean,
                    "std_seconds": full_timing.std,
                    "prepare_mean_seconds": prepare_once_timing.mean,
                    "prepare_std_seconds": prepare_once_timing.std,
                    "evaluate_mean_seconds": eval_timing.mean,
                    "evaluate_std_seconds": eval_timing.std,
                    "error": "",
                }
            )
        except Exception as exc:
            msg = f"{type(exc).__name__}: {exc}"
            print(f"[sweep_runtimes] N={num_particles} failed: {msg}")
            records.append(
                {
                    "num_particles": num_particles,
                    "mean_seconds": float("nan"),
                    "std_seconds": float("nan"),
                    "prepare_mean_seconds": float("nan"),
                    "prepare_std_seconds": float("nan"),
                    "evaluate_mean_seconds": float("nan"),
                    "evaluate_std_seconds": float("nan"),
                    "error": msg,
                }
            )
            if strict:
                raise
    if fmm is not None:
        _release_runtime_memory(fmm)
    return pd.DataFrame(records)


def profile_prepare_components_subprocess(
    particle_counts,
    *,
    leaf_size,
    max_order,
    dtype,
    key,
    fmm_kwargs,
    runs,
    warmup,
    strict=False,
):
    del key
    records = []
    for num_particles in particle_counts:
        try:
            out = _run_worker_case(
                "prepare",
                num_particles=int(num_particles),
                leaf_size=int(leaf_size),
                max_order=int(max_order),
                runs=int(runs),
                warmup=int(warmup),
                dtype=dtype,
                fmm_kwargs=fmm_kwargs,
            )
            records.append(
                {
                    "num_particles": int(num_particles),
                    "tree_build_mean_seconds": float(out.get("tree_build_mean_seconds", float("nan"))),
                    "upward_mean_seconds": float(out.get("upward_mean_seconds", float("nan"))),
                    "interactions_mean_seconds": float(out.get("interactions_mean_seconds", float("nan"))),
                    "downward_mean_seconds": float(out.get("downward_mean_seconds", float("nan"))),
                    "prepare_component_sum_seconds": float(out.get("prepare_component_sum_seconds", float("nan"))),
                    "error": str(out.get("error", "")),
                }
            )
        except Exception as exc:
            msg = f"{type(exc).__name__}: {exc}"
            print(f"[profile_prepare_components] N={num_particles} failed: {msg}")
            records.append(
                {
                    "num_particles": int(num_particles),
                    "tree_build_mean_seconds": float("nan"),
                    "upward_mean_seconds": float("nan"),
                    "interactions_mean_seconds": float("nan"),
                    "downward_mean_seconds": float("nan"),
                    "prepare_component_sum_seconds": float("nan"),
                    "error": msg,
                }
            )
            if strict:
                raise
    return pd.DataFrame(records)


def profile_prepare_components_runtime(
    particle_counts,
    *,
    leaf_size,
    max_order,
    dtype,
    key,
    fmm_kwargs,
    runs,
    warmup,
    strict=False,
):
    if not bool(globals().get("runtime_isolate_process_per_n", False)):
        return profile_prepare_components(
            particle_counts,
            leaf_size=leaf_size,
            max_order=max_order,
            dtype=dtype,
            key=key,
            fmm_kwargs=fmm_kwargs,
            runs=runs,
            warmup=warmup,
            strict=strict,
        )
    return profile_prepare_components_subprocess(
        particle_counts,
        leaf_size=leaf_size,
        max_order=max_order,
        dtype=dtype,
        key=key,
        fmm_kwargs=fmm_kwargs,
        runs=runs,
        warmup=warmup,
        strict=strict,
    )


## Operator accuracy diagnostics

These checks isolate accuracy scaling of individual operators (P2M/M2M/M2L/L2P) outside the full tree/MAC pipeline. They help confirm whether order improves accuracy for z-aligned and off-axis interactions.

In [ ]:
from jaccpot.operators.real_harmonics import (
    evaluate_local_real,
    m2l_real,
    m2m_real,
    p2m_real_direct,
    translate_along_z_m2l_real,
    sh_size,
 )

def m2m_error_vs_order(*, orders, child_center, parent_center, positions, masses):
    errors = []
    for order in orders:
        # direct P2M about parent center
        deltas_parent = positions - parent_center
        m_parent = jnp.sum(
            jax.vmap(lambda d, m: p2m_real_direct(d, m, order=order))(deltas_parent, masses),
            axis=0,
        )
        # P2M about child + M2M to parent
        deltas_child = positions - child_center
        m_child = jnp.sum(
            jax.vmap(lambda d, m: p2m_real_direct(d, m, order=order))(deltas_child, masses),
            axis=0,
        )
        delta_m2m = child_center - parent_center
        m_trans = m2m_real(m_child, delta_m2m, order=order)
        err = float(jnp.linalg.norm(m_trans - m_parent) / jnp.linalg.norm(m_parent))
        errors.append(err)
    return errors


def m2l_error_vs_order(*, orders, source_pos, local_center, eval_offset):
    errors = []
    for order in orders:
        multipole = p2m_real_direct(source_pos, jnp.asarray(1.0), order=order)
        # M2L uses delta = target - source
        delta_m2l = local_center - jnp.asarray([0.0, 0.0, 0.0])
        local = m2l_real(multipole, delta_m2l, order=order)
        eval_point = local_center + eval_offset
        delta_l2p = local_center - eval_point
        potential_fmm = evaluate_local_real(local, delta_l2p, order=order)
        r = jnp.linalg.norm(eval_point - source_pos)
        potential_direct = 1.0 / r
        err = float(jnp.abs(potential_fmm - potential_direct) / jnp.abs(potential_direct))
        errors.append(err)
    return errors


def m2l_error_zaxis_vs_order(*, orders, source_pos, local_center, eval_offset):
    errors = []
    for order in orders:
        multipole = p2m_real_direct(source_pos, jnp.asarray(1.0), order=order)
        R = jnp.linalg.norm(local_center - jnp.asarray([0.0, 0.0, 0.0]))
        local = translate_along_z_m2l_real(multipole, jnp.asarray(R), order=order)
        eval_point = local_center + eval_offset
        delta_l2p = local_center - eval_point
        potential_fmm = evaluate_local_real(local, delta_l2p, order=order)
        r = jnp.linalg.norm(eval_point - source_pos)
        potential_direct = 1.0 / r
        err = float(jnp.abs(potential_fmm - potential_direct) / jnp.abs(potential_direct))
        errors.append(err)
    return errors

In [ ]:
orders_diag = [1, 2, 3, 4, 5, 6]

# M2M check (two particles in child node)
child_center = jnp.array([0.2, -0.1, 0.3])
parent_center = jnp.array([0.0, 0.0, 0.0])
positions_m2m = jnp.array([[0.3, -0.05, 0.25], [0.1, -0.2, 0.45]])
masses_m2m = jnp.array([1.2, 0.7])
m2m_errors = m2m_error_vs_order(
    orders=orders_diag,
    child_center=child_center,
    parent_center=parent_center,
    positions=positions_m2m,
    masses=masses_m2m,
 )

# M2L on z-axis (translation-only, no rotation)
source_pos = jnp.array([0.0, 0.0, 0.0])
local_center_z = jnp.array([0.0, 0.0, 5.0])
eval_offset = jnp.array([0.2, -0.1, 0.15])
m2l_z_errors = m2l_error_zaxis_vs_order(
    orders=orders_diag,
    source_pos=source_pos,
    local_center=local_center_z,
    eval_offset=eval_offset,
 )

# M2L off-axis (full rotation path)
local_center_off = jnp.array([3.0, 2.0, 4.0])
m2l_off_errors = m2l_error_vs_order(
    orders=orders_diag,
    source_pos=source_pos,
    local_center=local_center_off,
    eval_offset=eval_offset,
 )

pd.DataFrame(
    {
        "order": orders_diag,
        "m2m_rel_err": m2m_errors,
        "m2l_zaxis_rel_err": m2l_z_errors,
        "m2l_offaxis_rel_err": m2l_off_errors,
    }
 )

## Expansion-order scaling (accuracy vs multipole order)

For a clean convergence curve, we fix the opening angle θ and sweep the multipole expansion order. This isolates the effect of truncation order on accuracy (at roughly fixed MAC aggressiveness).

Notes:
- Larger orders increase compute cost but should reduce error until other error sources (float precision, tree geometry, MAC) dominate.
- Keeping `order_sweep_particle_count` larger than the θ/order grid can reduce sampling noise in the error curves, but the direct reference is $O(N^2)$ so use with care.

In [ ]:
positions_order, masses_order, _ = bench_utils.generate_random_distribution(
    order_sweep_particle_count,
    key=order_diag_key,
    dtype=accuracy_working_dtype,
)

order_sweep_kwargs = override_fmm_kwargs(accuracy_fmm_kwargs, mac_type=order_sweep_mac_type)

order_sweep_df = sweep_accuracy(
    positions_order,
    masses_order,
    thetas=[order_sweep_theta],
    orders=order_sweep_orders,
    leaf_size=order_sweep_leaf_size,
    fmm_kwargs=order_sweep_kwargs,
)
order_sweep_df = order_sweep_df.sort_values("max_order")
order_sweep_df

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(
    order_sweep_df["max_order"],
    order_sweep_df["relative_l2_error"],
    marker="o",
    label="rel L2",
)
ax.semilogy(
    order_sweep_df["max_order"],
    order_sweep_df["relative_max_error"],
    marker="s",
    label="rel max",
)
ax.set_xlabel("Multipole expansion order")
ax.set_ylabel("Relative error")
ax.set_title(f"Accuracy scaling with order (θ={order_sweep_theta})")
ax.grid(True, which="both", linestyle=":", linewidth=0.8)
ax.legend()
plt.show()

## MAC sensitivity panel (theta × MAC type)
This panel isolates how much of the global error floor is driven by the MAC criterion at fixed expansion order.


In [ ]:
diag_particle_count = 1024
diag_leaf_size = 16
diag_mac_order = 4
diag_thetas = [0.10, 0.18, 0.20, 0.25, 0.30, 0.40]
diag_mac_types = ["bh", "engblom", "dehnen"]
diag_key = jax.random.PRNGKey(42)

positions_diag, masses_diag, _ = bench_utils.generate_random_distribution(
    diag_particle_count,
    key=diag_key,
    dtype=accuracy_working_dtype,
)
reference_diag = direct_accelerations(
    positions_diag,
    masses_diag,
    softening=softening,
)

mac_rows = []
for mac_type in diag_mac_types:
    for theta in diag_thetas:
        kwargs = override_fmm_kwargs(
            accuracy_fmm_kwargs, theta=float(theta), mac_type=mac_type
        )

        fmm = FastMultipoleMethod(**kwargs)
        accelerations = fmm.compute_accelerations(
            positions_diag,
            masses_diag,
            leaf_size=diag_leaf_size,
            max_order=diag_mac_order,
        )

        mac_rows.append(
            {
                "mac_type": mac_type,
                "theta": float(theta),
                "max_order": int(diag_mac_order),
                "relative_l2_error": relative_l2_error(accelerations, reference_diag),
                "relative_max_error": relative_max_error(accelerations, reference_diag),
            }
        )

mac_diag_df = pd.DataFrame(mac_rows).sort_values(["mac_type", "theta"])
mac_diag_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

for mac_type, group in mac_diag_df.groupby("mac_type"):
    g = group.sort_values("theta")
    axes[0].semilogy(g["theta"], g["relative_l2_error"], marker="o", label=mac_type)
    axes[1].semilogy(g["theta"], g["relative_max_error"], marker="s", label=mac_type)

axes[0].set_xlabel("theta")
axes[0].set_ylabel("Relative L2 error")
axes[0].set_title(f"MAC sensitivity @ order={diag_mac_order}")
axes[0].grid(True, which="both", linestyle=":", linewidth=0.8)
axes[0].legend(title="MAC")

axes[1].set_xlabel("theta")
axes[1].set_ylabel("Relative max error")
axes[1].set_title(f"MAC sensitivity @ order={diag_mac_order}")
axes[1].grid(True, which="both", linestyle=":", linewidth=0.8)
axes[1].legend(title="MAC")

plt.tight_layout()
plt.show()


## Accuracy sweep

Compare FMM outputs against a direct-sum reference for multiple opening angles (θ) and multipole expansion orders. Relative L2 error is reported on a log scale to highlight convergence trends. Values of θ≲0.3 keep the multipole acceptance criterion tight enough for high accuracy; larger θ intentionally trade error for speed.

In [ ]:
positions_accuracy, masses_accuracy, _ = bench_utils.generate_random_distribution(
    accuracy_particle_count,
    key=accuracy_key,
    dtype=accuracy_working_dtype,
 )
accuracy_df = sweep_accuracy(
    positions_accuracy,
    masses_accuracy,
    thetas=accuracy_thetas,
    orders=accuracy_orders,
    leaf_size=accuracy_leaf_size,
    fmm_kwargs=accuracy_fmm_kwargs,
 )
accuracy_df

### Numerical error table

Pivot the sweep results to inspect actual relative errors for each (θ, order) pair.

In [ ]:
accuracy_table = (
    accuracy_df.pivot_table(
        values="relative_l2_error",
        index="max_order",
        columns="theta",
        aggfunc="first",
    )
    .sort_index(axis=0)
    .sort_index(axis=1)
 )
accuracy_table.style.format("{:.2e}").set_caption(
    "Relative L2 error across θ and multipole order"
 )

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for order in accuracy_orders:
    subset = accuracy_df[accuracy_df["max_order"] == order].sort_values("theta")
    ax.semilogy(
        subset["theta"],
        subset["relative_l2_error"],
        marker="o",
        label=f"order {order}",
    )
ax.set_xlabel("Opening angle θ")
ax.set_ylabel("Relative L2 error")
ax.set_title("Accuracy vs θ and multipole order")
ax.grid(True, which="both", linestyle=":", linewidth=0.8)
ax.legend(title="Multipole order")
plt.show()

## Complex solidFMM operator accuracy (solidfmm rotations)
This section benchmarks accuracy of each complex operator (P2M, M2M, M2L, L2L, L2P) against direct potential or coefficient references, with solidfmm rotations enabled.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import pandas as pd

from jaccpot.operators.complex_harmonics import complex_R_solidfmm, complex_S_solidfmm, p2m_complex
from jaccpot.operators.complex_ops import (
    complex_dot,
    m2m_complex,
    m2l_complex_reference,
    l2l_complex,
    evaluate_local_complex,
 )

op_orders = list(range(1, 9))
dtype = jnp.float64

source_pos = jnp.array([0.2, -0.1, 0.3], dtype=dtype)
multipole_center = jnp.array([0.0, 0.0, 0.0], dtype=dtype)
parent_center = jnp.array([0.0, 0.0, 0.0], dtype=dtype)
child_center = jnp.array([0.4, -0.2, 0.3], dtype=dtype)
local_center = jnp.array([3.0, 2.0, 4.0], dtype=dtype)
eval_offset = jnp.array([0.2, -0.1, 0.15], dtype=dtype)
eval_offset_child = jnp.array([-0.1, 0.25, 0.05], dtype=dtype)

positions_m2m = jnp.array([[0.3, -0.05, 0.25], [0.1, -0.2, 0.45]], dtype=dtype)
masses_m2m = jnp.array([1.2, 0.7], dtype=dtype)

def p2m_sum(positions, masses, center, *, order):
    deltas = positions - center
    return jnp.sum(
        jax.vmap(lambda d, m: p2m_complex(d, m, order=order))(deltas, masses),
        axis=0,
    )

def multipole_potential(multipole, center, eval_point, *, order):
    delta = eval_point - center
    sing = complex_S_solidfmm(delta, order=order)
    pot = complex_dot(multipole, sing, order=order, conjugate_left=True)
    return jnp.real(pot)

def direct_potential(source, eval_point, mass=1.0):
    r = jnp.linalg.norm(eval_point - source)
    return mass / r

errors = {"P2M": [], "M2M": [], "M2L": [], "L2L": [], "L2P": []}

for order in op_orders:
    # P2M: multipole -> potential vs direct
    multipole = p2m_complex(source_pos - multipole_center, jnp.asarray(1.0, dtype=dtype), order=order)
    eval_point = local_center + eval_offset
    pot_p2m = multipole_potential(multipole, multipole_center, eval_point, order=order)
    pot_direct = direct_potential(source_pos, eval_point)
    errors["P2M"].append(float(jnp.abs(pot_p2m - pot_direct) / jnp.abs(pot_direct)))

    # M2M: compare coefficients (parent direct vs child-translated)
    m_parent = p2m_sum(positions_m2m, masses_m2m, parent_center, order=order)
    m_child = p2m_sum(positions_m2m, masses_m2m, child_center, order=order)
    m_trans = m2m_complex(
        m_child, child_center - parent_center, order=order, rotation="solidfmm"
    )
    errors["M2M"].append(float(jnp.linalg.norm(m_trans - m_parent) / jnp.linalg.norm(m_parent)))

    # M2L: multipole -> local -> potential vs direct
    multipole = p2m_complex(source_pos - multipole_center, jnp.asarray(1.0, dtype=dtype), order=order)
    local = m2l_complex_reference(
        multipole,
        local_center - multipole_center,
        order=order,
        rotation="solidfmm",
    )
    eval_point = local_center + eval_offset
    pot_m2l = evaluate_local_complex(
        local,
        local_center - eval_point,
        order=order,
        conjugate_left=True,
    )
    pot_direct = direct_potential(source_pos, eval_point)
    errors["M2L"].append(float(jnp.abs(pot_m2l - pot_direct) / jnp.abs(pot_direct)))

    # L2L: parent local -> child local -> potential vs direct
    local_child = l2l_complex(
        local, child_center - local_center, order=order, rotation="solidfmm",
    )
    eval_point = child_center + eval_offset_child
    pot_l2l = evaluate_local_complex(
        local_child,
        child_center - eval_point,
        order=order,
        conjugate_left=True,
    )
    pot_direct = direct_potential(source_pos, eval_point)
    errors["L2L"].append(float(jnp.abs(pot_l2l - pot_direct) / jnp.abs(pot_direct)))

    # L2P: evaluate local vs direct (same local as M2L)
    eval_point = local_center + eval_offset_child
    pot_l2p = evaluate_local_complex(
        local,
        local_center - eval_point,
        order=order,
        conjugate_left=True,
    )
    pot_direct = direct_potential(source_pos, eval_point)
    errors["L2P"].append(float(jnp.abs(pot_l2p - pot_direct) / jnp.abs(pot_direct)))

errors_df = pd.DataFrame({"order": op_orders, **errors})
errors_df

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for key in ["P2M", "M2M", "M2L", "L2L", "L2P"]:
    ax.semilogy(errors_df["order"], errors_df[key], marker="o", label=key)
ax.set_xlabel("Expansion order p")
ax.set_ylabel("Relative error")
ax.set_title("Complex solidFMM operator accuracy (solidfmm rotations)")
ax.grid(True, which="both", linestyle=":", linewidth=0.8)
ax.legend()
plt.show()

In [ ]:
# Diagnostics: ensure geometry is in convergence region for L2L/L2P
diag_orders = list(range(1, 9))
dtype = jnp.float64

source_pos = jnp.array([0.25, -0.15, 0.35], dtype=dtype)
multipole_center = jnp.array([0.0, 0.0, 0.0], dtype=dtype)
local_center = jnp.array([4.0, 3.0, 5.0], dtype=dtype)
child_center = local_center + jnp.array([0.3, -0.2, 0.1], dtype=dtype)
eval_offset = jnp.array([0.1, -0.05, 0.08], dtype=dtype)

positions_m2m = jnp.array([[0.3, -0.05, 0.25], [0.1, -0.2, 0.45]], dtype=dtype)
masses_m2m = jnp.array([1.2, 0.7], dtype=dtype)
parent_center = jnp.array([0.0, 0.0, 0.0], dtype=dtype)
child_center_m2m = jnp.array([0.4, -0.2, 0.3], dtype=dtype)

def p2m_sum_complex(positions, masses, center, *, order):
    deltas = positions - center
    return jnp.sum(
        jax.vmap(lambda d, m: p2m_complex(d, m, order=order))(deltas, masses),
        axis=0,
    )

def direct_potential(source, eval_point, mass=1.0):
    r = jnp.linalg.norm(eval_point - source)
    return mass / r

diag_errors = {"M2M": [], "L2L": [], "L2P": []}

for order in diag_orders:
    # M2M coefficient consistency (child -> parent)
    m_parent = p2m_sum_complex(positions_m2m, masses_m2m, parent_center, order=order)
    m_child = p2m_sum_complex(positions_m2m, masses_m2m, child_center_m2m, order=order)
    m_trans = m2m_complex(m_child, child_center_m2m - parent_center, order=order, rotation="solidfmm")
    diag_errors["M2M"].append(float(jnp.linalg.norm(m_trans - m_parent) / jnp.linalg.norm(m_parent)))

    # Build local via M2L (well separated)
    multipole = p2m_complex(source_pos - multipole_center, jnp.asarray(1.0, dtype=dtype), order=order)
    local = m2l_complex_reference(
        multipole,
        local_center - multipole_center,
        order=order,
        rotation="solidfmm",
    )

    # L2L: translate local to a nearby child center and evaluate
    local_child = l2l_complex(
        local,
        child_center - local_center,
        order=order,
        rotation="solidfmm",
    )
    eval_point = child_center + eval_offset
    pot_l2l = evaluate_local_complex(
        local_child,
        child_center - eval_point,
        order=order,
        conjugate_left=True,
    )
    pot_direct = direct_potential(source_pos, eval_point)
    diag_errors["L2L"].append(float(jnp.abs(pot_l2l - pot_direct) / jnp.abs(pot_direct)))

    # L2P: evaluate local near its own center
    eval_point = local_center + eval_offset
    pot_l2p = evaluate_local_complex(
        local,
        local_center - eval_point,
        order=order,
        conjugate_left=True,
    )
    pot_direct = direct_potential(source_pos, eval_point)
    diag_errors["L2P"].append(float(jnp.abs(pot_l2p - pot_direct) / jnp.abs(pot_direct)))

diag_df = pd.DataFrame({"order": diag_orders, **diag_errors})
diag_df

### Geometry convergence check
Verify that evaluation points are inside the convergence balls of the local expansions. For a local expansion centered at $c$, the convergence radius is $|c-\text{source}|$; evaluation points should satisfy $|x-c| < |c-\text{source}|$.

In [ ]:
def _add_geom_case(rows, label, source, center, eval_point):
    r_source = float(jnp.linalg.norm(source - center))
    r_eval = float(jnp.linalg.norm(eval_point - center))
    rows.append(
        {
            "case": label,
            "r_source": r_source,
            "r_eval": r_eval,
            "ratio_r_eval_over_r_source": r_eval / r_source if r_source > 0 else float("inf"),
            "inside_convergence": r_eval < r_source,
        }
    )

geom_rows = []

if "source_pos" in globals() and "local_center" in globals() and "eval_offset" in globals():
    _add_geom_case(
        geom_rows,
        "parent_local_eval_offset",
        source_pos,
        local_center,
        local_center + eval_offset,
    )

if "source_pos" in globals() and "child_center" in globals() and "eval_offset" in globals():
    _add_geom_case(
        geom_rows,
        "child_local_eval_offset",
        source_pos,
        child_center,
        child_center + eval_offset,
    )

if "source_pos" in globals() and "child_center" in globals() and "eval_offset_child" in globals():
    _add_geom_case(
        geom_rows,
        "child_local_eval_offset_child",
        source_pos,
        child_center,
        child_center + eval_offset_child,
    )

if "source_pos" in globals() and "local_center" in globals() and "eval_offset_child" in globals():
    _add_geom_case(
        geom_rows,
        "parent_local_eval_offset_child",
        source_pos,
        local_center,
        local_center + eval_offset_child,
    )

if "source_pos" in globals() and "child_local_center" in globals() and "eval_offset" in globals():
    _add_geom_case(
        geom_rows,
        "child_local_center_eval_offset",
        source_pos,
        child_local_center,
        child_local_center + eval_offset,
    )

pd.DataFrame(geom_rows)

### SolidFMM formula reference check (complex basis)
Compare solidfmm-rotation M2M/L2L against direct formulas using $R_n^m$ (no rotations), and then compare L2P on the translated locals.

In [ ]:
import jax
import jax.numpy as jnp
import pandas as pd
import importlib

from jaccpot.operators.complex_harmonics import complex_R_solidfmm, p2m_complex
import jaccpot.operators.complex_ops as complex_ops
importlib.reload(complex_ops)
from jaccpot.operators.complex_ops import (
    enforce_conjugate_symmetry,
    m2m_complex,
    l2l_complex,
    evaluate_local_complex,
 )
from jaccpot.operators.real_harmonics import sh_index, sh_size

def _rel_err(a, b):
    return float(jnp.linalg.norm(a - b) / jnp.linalg.norm(b))

def m2m_reference_complex(multipole, delta, *, order):
    p = int(order)
    multipole = jnp.asarray(multipole)
    R = complex_R_solidfmm(delta, order=p)
    out = jnp.zeros((sh_size(p),), dtype=multipole.dtype)
    for n in range(p + 1):
        for m in range(-n, n + 1):
            acc = 0.0 + 0.0j
            for k in range(p + 1):
                if n - k < 0:
                    continue
                for l in range(-k, k + 1):
                    q = n - k
                    t = m - l
                    if abs(t) > q:
                        continue
                    acc = acc + multipole[sh_index(k, l)] * R[sh_index(q, t)]
            out = out.at[sh_index(n, m)].set(acc)
    return out

def l2l_reference_complex(local, delta, *, order):
    p = int(order)
    local = jnp.asarray(local)
    R = complex_R_solidfmm(delta, order=p)
    out = jnp.zeros((sh_size(p),), dtype=local.dtype)
    for n in range(p + 1):
        for m in range(-n, n + 1):
            acc = 0.0 + 0.0j
            for k in range(n, p + 1):
                for l in range(-k, k + 1):
                    q = k - n
                    t = l - m
                    if abs(t) > q:
                        continue
                    acc = acc + local[sh_index(k, l)] * jnp.conjugate(R[sh_index(q, t)])
            out = out.at[sh_index(n, m)].set(acc)
    return out

orders_ref = list(range(1, 8))
dtype = jnp.float64
delta_ref = jnp.array([0.35, -0.25, 0.45], dtype=dtype)
eval_offset = jnp.array([0.02, -0.01, 0.03], dtype=dtype)

# Use P2M to generate physical multipoles and locals (conjugate-symmetric by construction)
positions = jnp.array([[0.3, -0.05, 0.25], [0.1, -0.2, 0.45]], dtype=dtype)
masses = jnp.array([1.2, 0.7], dtype=dtype)

def p2m_sum(positions, masses, center, *, order):
    deltas = positions - center
    return jnp.sum(
        jax.vmap(lambda d, m: p2m_complex(d, m, order=order))(deltas, masses),
        axis=0,
    )


rows = []
for order in orders_ref:
    multipole = p2m_sum(positions, masses, jnp.zeros(3, dtype=dtype), order=order)
    local = p2m_sum(positions, masses, jnp.zeros(3, dtype=dtype), order=order)  # for L2L, just use same as multipole

    m2m_ref = m2m_reference_complex(multipole, delta_ref, order=order)
    l2l_ref = l2l_reference_complex(local, delta_ref, order=order)

    m2m_cached = m2m_complex(multipole, delta_ref, order=order, rotation="cached")
    m2m_wigner = m2m_complex(multipole, delta_ref, order=order, rotation="wigner")
    m2m_solidfmm = m2m_complex(multipole, delta_ref, order=order, rotation="solidfmm")

    l2l_cached = l2l_complex(local, delta_ref, order=order, rotation="cached")
    l2l_wigner = l2l_complex(local, delta_ref, order=order, rotation="wigner")
    l2l_solidfmm = l2l_complex(local, delta_ref, order=order, rotation="solidfmm")

    pot_cached = evaluate_local_complex(l2l_cached, eval_offset, order=order, conjugate_left=True)
    pot_wigner = evaluate_local_complex(l2l_wigner, eval_offset, order=order, conjugate_left=True)
    pot_solidfmm = evaluate_local_complex(l2l_solidfmm, eval_offset, order=order, conjugate_left=True)
    pot_ref = evaluate_local_complex(l2l_ref, eval_offset, order=order, conjugate_left=True)

    rows.append(
        {
            "order": order,
            "m2m_cached_rel_err": _rel_err(m2m_cached, m2m_ref),
            "m2m_wigner_rel_err": _rel_err(m2m_wigner, m2m_ref),
            "m2m_solidfmm_rel_err": _rel_err(m2m_solidfmm, m2m_ref),
            "l2l_cached_rel_err": _rel_err(l2l_cached, l2l_ref),
            "l2l_wigner_rel_err": _rel_err(l2l_wigner, l2l_ref),
            "l2l_solidfmm_rel_err": _rel_err(l2l_solidfmm, l2l_ref),
            "l2p_cached_rel_err": float(jnp.abs(pot_cached - pot_ref) / jnp.maximum(jnp.abs(pot_ref), 1e-30)),
            "l2p_wigner_rel_err": float(jnp.abs(pot_wigner - pot_ref) / jnp.maximum(jnp.abs(pot_ref), 1e-30)),
            "l2p_solidfmm_rel_err": float(jnp.abs(pot_solidfmm - pot_ref) / jnp.maximum(jnp.abs(pot_ref), 1e-30)),
        },
    )

pd.DataFrame(rows)

In [ ]:
# Solidfmm M2M check with conjugate-symmetric multipoles (P2M).
import jax
import jax.numpy as jnp
import pandas as pd

from jaccpot.operators.complex_harmonics import p2m_complex
from jaccpot.operators.complex_ops import m2m_complex

orders_sym = list(range(1, 8))
dtype = jnp.float64
parent_center = jnp.array([0.0, 0.0, 0.0], dtype=dtype)
child_center = jnp.array([0.4, -0.2, 0.3], dtype=dtype)
positions = jnp.array([[0.3, -0.05, 0.25], [0.1, -0.2, 0.45]], dtype=dtype)
masses = jnp.array([1.2, 0.7], dtype=dtype)

def p2m_sum_sym(positions, masses, center, *, order):
    deltas = positions - center
    return jnp.sum(
        jax.vmap(lambda d, m: p2m_complex(d, m, order=order))(deltas, masses),
        axis=0,
    )

rows = []
delta_m2m = child_center - parent_center
for order in orders_sym:
    m_parent = p2m_sum_sym(positions, masses, parent_center, order=order)
    m_child = p2m_sum_sym(positions, masses, child_center, order=order)
    m_trans = m2m_complex(
        m_child,
        delta_m2m,
        order=order,
        rotation="solidfmm",
    )
    m_trans_flip = m2m_complex(
        m_child,
        -delta_m2m,
        order=order,
        rotation="solidfmm",
    )
    err = float(jnp.linalg.norm(m_trans - m_parent) / jnp.linalg.norm(m_parent))
    err_flip = float(
        jnp.linalg.norm(m_trans_flip - m_parent) / jnp.linalg.norm(m_parent)
    )
    rows.append(
        {
            "order": order,
            "m2m_solidfmm_rel_err": err,
            "m2m_solidfmm_rel_err_flip": err_flip,
        }
    )

pd.DataFrame(rows)

In [ ]:
# Multipole rotation invariance check (potential should be rotation-invariant).
import jax
import jax.numpy as jnp
import pandas as pd

from jaccpot.operators.complex_harmonics import complex_S_solidfmm
from jaccpot.operators.complex_ops import (
    complex_dot,
    rotate_complex_multipole_to_z_cached,
    rotate_complex_multipole_to_z_solidfmm,
    rotate_complex_multipole_to_z_wigner,
 )
from jaccpot.operators.real_harmonics import sh_size

orders_rot = list(range(1, 8))
dtype = jnp.float64
delta_rot = jnp.array([0.35, -0.25, 0.45], dtype=dtype)
r_rot = jnp.linalg.norm(delta_rot)
delta_z = jnp.array([0.0, 0.0, r_rot], dtype=dtype)

rows_rot = []
for order in orders_rot:
    key = jax.random.PRNGKey(100 + order)
    multipole = (
        jax.random.normal(key, (sh_size(order),), dtype=dtype)
        + 1j * jax.random.normal(key, (sh_size(order),), dtype=dtype)
    )
    sing_orig = complex_S_solidfmm(delta_rot, order=order)
    sing_z = complex_S_solidfmm(delta_z, order=order)
    pot_orig = jnp.real(complex_dot(multipole, sing_orig, order=order, conjugate_left=True))

    mp_cached = rotate_complex_multipole_to_z_cached(multipole, delta_rot, order=order)
    mp_wigner = rotate_complex_multipole_to_z_wigner(multipole, delta_rot, order=order)
    mp_solidfmm = rotate_complex_multipole_to_z_solidfmm(multipole, delta_rot, order=order)

    pot_cached = jnp.real(complex_dot(mp_cached, sing_z, order=order, conjugate_left=True))
    pot_wigner = jnp.real(complex_dot(mp_wigner, sing_z, order=order, conjugate_left=True))
    pot_solidfmm = jnp.real(complex_dot(mp_solidfmm, sing_z, order=order, conjugate_left=True))

    denom = jnp.maximum(jnp.abs(pot_orig), 1e-30)
    rows_rot.append(
        {
            "order": order,
            "cached_rel_err": float(jnp.abs(pot_cached - pot_orig) / denom),
            "wigner_rel_err": float(jnp.abs(pot_wigner - pot_orig) / denom),
            "solidfmm_rel_err": float(jnp.abs(pot_solidfmm - pot_orig) / denom),
        }
    )

pd.DataFrame(rows_rot)

In [ ]:
# Local rotation invariance check (potential should be rotation-invariant).
import jax
import jax.numpy as jnp
import pandas as pd

from jaccpot.operators.complex_harmonics import complex_R_solidfmm
from jaccpot.operators.complex_ops import (
    complex_dot,
    rotate_complex_local_to_z_cached,
    rotate_complex_local_to_z_solidfmm,
    rotate_complex_local_to_z_wigner,
 )
from jaccpot.operators.real_harmonics import sh_size

orders_loc = list(range(1, 8))
dtype = jnp.float64
delta_loc = jnp.array([0.35, -0.25, 0.45], dtype=dtype)
r_loc = jnp.linalg.norm(delta_loc)
delta_z_loc = jnp.array([0.0, 0.0, r_loc], dtype=dtype)

rows_loc = []
for order in orders_loc:
    key = jax.random.PRNGKey(200 + order)
    local = (
        jax.random.normal(key, (sh_size(order),), dtype=dtype)
        + 1j * jax.random.normal(key, (sh_size(order),), dtype=dtype)
    )
    reg_orig = complex_R_solidfmm(delta_loc, order=order)
    reg_z = complex_R_solidfmm(delta_z_loc, order=order)
    pot_orig = jnp.real(complex_dot(local, reg_orig, order=order, conjugate_left=True))

    loc_cached = rotate_complex_local_to_z_cached(local, delta_loc, order=order)
    loc_wigner = rotate_complex_local_to_z_wigner(local, delta_loc, order=order)
    loc_solidfmm = rotate_complex_local_to_z_solidfmm(local, delta_loc, order=order)

    pot_cached = jnp.real(complex_dot(loc_cached, reg_z, order=order, conjugate_left=True))
    pot_wigner = jnp.real(complex_dot(loc_wigner, reg_z, order=order, conjugate_left=True))
    pot_solidfmm = jnp.real(complex_dot(loc_solidfmm, reg_z, order=order, conjugate_left=True))

    denom = jnp.maximum(jnp.abs(pot_orig), 1e-30)
    rows_loc.append(
        {
            "order": order,
            "cached_rel_err": float(jnp.abs(pot_cached - pot_orig) / denom),
            "wigner_rel_err": float(jnp.abs(pot_wigner - pot_orig) / denom),
            "solidfmm_rel_err": float(jnp.abs(pot_solidfmm - pot_orig) / denom),
        }
    )

pd.DataFrame(rows_loc)